# Phase 1 — Build the 512 px cache

Decodes every fundus image once instead of every epoch. This is the difference
between a ~4 hour training run and a ~1.5 hour one, and therefore between a feasible
thesis and an infeasible one.

## Notebook settings (right-hand panel)

| Setting | Value | Why it matters |
|---|---|---|
| Accelerator | **None** | This job is pure CPU. Attaching a GPU burns a third of your weekly quota on JPEG decoding. |
| Persistence | **Files only** | `build_cache.py` resumes. Persistence lets a 10-hour build span several sittings with zero rework. |
| Internet | On (git clone only) | cv2 and numpy are preinstalled; nothing else is needed. |
| Environment | **Pin to original environment** | Kaggle updates its base image. Over 20 weeks that will silently change something. |

## Inputs

EyePACS, DDR, IDRiD now. APTOS and Messidor-2 can come later — they aren't touched
until Phase 6.

## How to run this

Work through **step 1 interactively and look at the contact sheet**. Only once the
crops look right, run the rest with *Save & Run All (Commit)* so it continues
unattended — interactive sessions die when your browser idles.

## Exit condition

`cache_report.json` for each dataset with a crop fallback rate ≤ 0.005, and the cache
saved as a Kaggle dataset. **Never rebuild it after Phase 2** — every downstream
number assumes a fixed cache.

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"
print("repo ready at", REPO_DIR)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook."""
    for d in sorted(INPUT.iterdir()):
        name = d.name.lower().replace("-", "").replace("_", "")
        if all(k.lower().replace("-", "").replace("_", "") in name for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print(f"     Use 'Add Input' in the right-hand panel. Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

CACHE = Path("/kaggle/working/cache512")
SCRIPT = REPO_DIR / "scripts/build_cache.py"
print("cache ->", CACHE)

---
## 2 · Smoke test — 200 images, then look

Never launch a ten-hour build on an unverified crop. This takes about a minute.

In [ ]:
eyepacs = find_mount("eyepacs")
cmd = (f"python {SCRIPT} --source-root {eyepacs} --dataset EyePACS "
       f"--output-root {CACHE} --limit 200 --contact-sheet 100")
!{cmd}

In [ ]:
from IPython.display import Image, display
display(Image(str(CACHE / "eyepacs/contact_sheet.jpg")))

### ⚠️ Stop here and actually look at that sheet

**Good:** each retina fills its tile, roughly circular, tightly framed, centred. A
little black in the corners is expected — `--fit pad` keeps the full field on purpose,
because M3's quadrant rules need the periphery.

**Bad, and what it means:**

| What you see | Cause | Fix |
|---|---|---|
| Wide black bars left/right | Crop didn't fire; surround noise defeated the threshold | Raise `--tol-scale` to 0.15–0.20 and re-run |
| Retina cut off at the edges | Crop too aggressive | Lower `--tol-scale` to 0.05 |
| Mostly-black or blank tiles | Genuinely ungradable images | Expected in EyePACS — quantify, don't fix |
| Washed-out, flat contrast | CLAHE too strong for this source | Lower `--clahe-clip` to 1.5, or `--no-clahe` |

Also check the printed **crop fallback rate**. A0's gate is ≤ 0.005; the script flags
it when exceeded. A high rate on the smoke sample is worth understanding *now*.

Re-run the cell above until the sheet looks right. Only then continue.

---
## 3 · Full EyePACS build

Around 45–90 minutes depending on how many images the mirror carries. Resumable — if
the session dies, just re-run this cell. **Never pass `--overwrite`**, that throws
away completed work.

In [ ]:
cmd = (f"python {SCRIPT} --source-root {eyepacs} --dataset EyePACS "
       f"--output-root {CACHE} --contact-sheet 100")
!{cmd}

---
## 4 · DDR — grading images and the lesion masks

Two passes. The first caches the grading images; the second caches the segmentation
subset **with its masks**, which is what M2 trains on.

Mask paths come from `verification_log.json` written by `00_verify_inputs.ipynb`, so
this adapts to whatever layout your DDR mirror actually uses.

In [ ]:
ddr = find_mount("ddr")
grading = next((p for p in ddr.rglob("*")
                if p.is_dir() and "grading" in p.name.lower()), ddr)
print("DDR grading images:", grading)

cmd = (f"python {SCRIPT} --source-root {grading} --dataset DDR "
       f"--output-root {CACHE} --contact-sheet 60")
!{cmd}

In [ ]:
# Rebuild the mask mapping (or load it from Phase 0's verification_log.json).
DDR_CHANNELS = {"MA": "microaneurysm", "HE": "haemorrhage",
                "EX": "hard_exudate", "SE": "soft_exudate"}

log = Path("/kaggle/working/verification_log.json")
mask_dirs = json.loads(log.read_text())["ddr_mask_dirs"] if log.exists() else {}
if not mask_dirs:
    for code_, channel in DDR_CHANNELS.items():
        hits = [str(p) for p in ddr.rglob(code_)
                if p.is_dir() and "segmentation" in str(p).lower()]
        if hits:
            mask_dirs[channel] = hits

seg_images = [p for p in ddr.rglob("*") if p.is_dir()
              and p.name.lower() == "image" and "segmentation" in str(p).lower()]

print("mask channels:", list(mask_dirs))
print("segmentation image dirs:", [str(p) for p in seg_images])
assert mask_dirs and seg_images, "DDR lesion masks not found - re-run 00_verify_inputs.ipynb"

In [ ]:
# One pass per split directory (train/valid/test), each with its own mask folders.
for img_dir in seg_images:
    split = img_dir.parent.name
    args = []
    for channel, dirs in mask_dirs.items():
        match = [d for d in dirs if f"/{split}/" in d + "/"]
        if match:
            args += ["--mask", f"{channel}={match[0]}"]
    if not args:
        print(f"skip {split}: no matching mask dirs")
        continue
    print(f"\n=== DDR segmentation split: {split} ===")
    cmd = (f"python {SCRIPT} --source-root {img_dir} --dataset DDR "
           f"--output-root {CACHE} --contact-sheet 0 " + " ".join(args))
    !{cmd}

---
## 5 · IDRiD — lesion masks and geometry

Small (516 images) and quick. The coordinate CSVs aren't cached — `prepare_manifest.py`
reads them directly in Phase 2.

In [ ]:
idrid = find_mount("idrid")
cmd = (f"python {SCRIPT} --source-root {idrid} --dataset IDRiD "
       f"--output-root {CACHE} --contact-sheet 60")
!{cmd}

---
## 6 · APTOS and Messidor-2 — *optional now*

These are the **locked external sets**. Caching them is harmless — it's a decode, not
an evaluation — but nothing may read them until after the Phase 5 freeze
(`docs/02_research_protocol.md` Rule 1).

Skip this cell if you'd rather not have them on disk at all until Phase 6.

In [ ]:
for keywords, name in [(("aptos",), "APTOS"), (("messidor2preprocess",), "Messidor2")]:
    mount = find_mount(*keywords, required=False)
    if mount is None:
        print(f"{name}: not mounted, skipping")
        continue
    src = next((p for p in mount.rglob("*") if p.is_dir()
                and "train" in p.name.lower() and "image" in p.name.lower()), mount)
    print(f"\n=== {name} from {src} ===")
    cmd = (f"python {SCRIPT} --source-root {src} --dataset {name} "
           f"--output-root {CACHE} --contact-sheet 60")
    !{cmd}

---
## 7 · Experiment A0 — the gate

A0 passes when counts reconcile, the crop fallback rate is ≤ 0.005 everywhere, and the
contact sheets look right. Record the result in `docs/04_experiment_register.md`.

In [ ]:
rows = []
for report in sorted(CACHE.rglob("cache_report.json")):
    r = json.loads(report.read_text())
    rows.append(r)

print(f"{'dataset':<12}{'found':>8}{'cached':>8}{'failed':>8}{'fallback':>11}{'MiB':>9}{'runs':>6}  gate")
print("-" * 76)
a0 = True
for r in rows:
    found, cached = r["counts"]["found"], r["cached"]
    failed, rate = r["counts"]["failed"], r["crop"]["fallback_rate"]
    mib = r["output"]["total_mib"]
    # A0: everything cached, crop fallback under 0.005, failures under 0.1%
    ok = (cached >= found - max(1, 0.001 * found)
          and rate is not None and rate <= 0.005)
    a0 &= ok
    shown = f"{rate:.4f}" if rate is not None else "    n/a"
    print(f"{r['dataset']:<12}{found:>8}{cached:>8}{failed:>8}{shown:>11}"
          f"{mib:>9.0f}{r['runs']:>6}  {'PASS' if ok else 'FAIL'}")
    for channel, stat in r["masks"].items():
        if stat["written"]:
            print(f"    mask {channel:<16} written={stat['written']}")
    if r["failures"]:
        print(f"    first failure: {r['failures'][0]['path']} - {r['failures'][0]['error']}")

total = sum(r["output"]["total_mib"] for r in rows)
print("-" * 76)
print(f"total cache: {total:.0f} MiB")
print("\nA0", "PASS - proceed to Phase 2" if a0 else "FAIL - investigate before continuing")
print("Now eyeball each contact sheet below, then record A0 in docs/04_experiment_register.md.")

In [ ]:
from IPython.display import Image, display
for sheet in sorted(CACHE.rglob("contact_sheet.jpg")):
    print(sheet.parent.name)
    display(Image(str(sheet)))

---
## 8 · Save the cache

**Save Version → Save & Run All (Commit).** When it finishes, open the notebook's
Output tab and use *New Dataset* to publish `/kaggle/working/cache512` as a private
dataset named **`verify-dr-cache-512`**.

Every notebook from Phase 2 on takes that dataset as input and never touches the raw
sources again.

> **Freeze it.** Rebuilding the cache after Phase 2 invalidates every split,
> checkpoint and number that came from it. If you genuinely must rebuild, treat it as
> a protocol deviation and record it in `preregistration/PREREGISTRATION.md`.

**Next:** `02_manifests.ipynb` — blocked until `prepare_manifest.py` and
`build_variants.py` are written.